# Lektion 18: Absicherung von KI-Agenten mit kryptografischen Quittungen

## Praktisches Notizbuch

Dieses Notizbuch führt durch vier Übungen:

1. **Unterschreiben Sie Ihre erste Quittung** für einen Agentenwerkzeugaufruf und überprüfen Sie sie.
2. **Manipulieren Sie die Quittung** und beobachten Sie, wie die Überprüfung fehlschlägt.
3. **Erstellen Sie eine dreifache Quittungskette** und bestätigen Sie die Integrität der Kette.
4. **Verpacken Sie einen Microsoft Agent Framework-Werkzeugaufruf** so, dass jede Aktion eine Quittung ausgibt.

Alle kryptografischen Primitiven werden aus gut gepflegten Bibliotheken importiert (`pynacl` für Ed25519, `jcs` für RFC 8785 kanonisches JSON, `hashlib` aus der Python-Standardbibliothek für SHA-256). Die Quittungslogik selbst ist reines Python, das Sie lesen und ändern können.

Führen Sie die Zellen der Reihe nach aus. Jeder Abschnitt ist kurz und in sich geschlossen.


## Einrichtung

Installieren Sie die beiden Abhängigkeiten. Beide haben permissive Lizenzen (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Hilfsprogramme

Diese beiden Helfer behandeln die Base64url-Codierung (ohne Padding) und das SHA-256-Hashing beliebiger Objekte. Sie halten den Rest des Notebooks auf die Beleglogik selbst konzentriert.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Abschnitt 1: Unterzeichnen Sie Ihre erste Quittung

Stellen Sie sich vor, unser Mitarbeiter von **Contoso Travel** hat gerade Flüge von Sydney nach Los Angeles für einen Kunden gesucht. Wir möchten diesen Toolaufruf als unterschriebene Quittung aufzeichnen, damit ein zukünftiger Prüfer ihn ohne Vertrauen in uns verifizieren kann.

### Schritt 1.1: Generieren Sie einen Signaturschlüssel

In der Produktion würde der Signaturschlüssel des Mitarbeiters in einem Hardware-Sicherheitsmodul (HSM), Azure Key Vault oder einem ähnlichen geschützten Speicher leben. Für diese Lektion generieren wir einen frischen Schlüssel im Arbeitsspeicher.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Schritt 1.2: Erstelle die Liefernachweis-Nutzlast

Die Nutzlast enthält alles, wofür der Liefernachweis Beleg erbringen soll: wer gehandelt hat, welches Werkzeug, mit welchen Argumenten, was zurückkam, unter welcher Richtlinie und wann. Wir hashen die Argumente und das Ergebnis, anstatt sie inline einzufügen, damit der Liefernachweis keine sensiblen Inhalte preisgibt.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Schritt 1.3: Unterschreiben und zusammenfügen der Quittung

Drei Schritte:

1. Den Payload mit JCS kanonisieren, sodass zwei Implementierungen, die dieselbe logische Quittung erzeugen, byte-identische Bytes liefern.
2. Die kanonischen Bytes mit SHA-256 hashen.
3. Den Hash mit dem Ed25519-Privatschlüssel unterschreiben.

Die Signatur wird dann an den Original-Payload angehängt, um die endgültige Quittung zu erzeugen.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Schritt 1.4: Beleg verifizieren

Die Verifikation kehrt den Prozess um. Wir entfernen die Signatur, berechnen den kanonischen Hash neu und überprüfen die Signatur anhand des öffentlichen Schlüssels im Beleg.

Ein Prüfer, der diese Verifikation durchführt, benötigt von uns nur den Beleg selbst. Kein Dienst muss aufgerufen werden, kein Schlüsseldirectory abgefragt, kein Vertrauen vorausgesetzt.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Sie sollten `Beleg ist gültig: Wahr` sehen. Der Agent hat seinen ersten kryptographisch signierten Prüfungsdatensatz erstellt.


## Abschnitt 2: Am Beleg manipulieren

Der ganze Sinn von Belegen ist, dass sie manipulationssicher sind. Lassen Sie es uns beweisen.

Wir werden genau ein Zeichen des Belegs ändern und beobachten, wie die Überprüfung fehlschlägt.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Was ist gerade passiert?

Als wir `policy_id` geändert haben, änderten sich die kanonischen Bytes. Der SHA-256-Hash dieser Bytes änderte sich. Die Signatur (die über den ursprünglichen Hash erstellt wurde) stimmt nicht mehr mit dem neuen Hash überein. Die Verifikation gibt korrekt `False` zurück.

Es gibt keine Möglichkeit, ein Feld der Quittung zu ändern und trotzdem eine gültige Verifizierung zu erreichen, es sei denn, der Angreifer besitzt den privaten Schlüssel. Solange der private Schlüssel in einem Schlüsseltresor aufbewahrt wird und der öffentliche Schlüssel veröffentlicht ist, ist eine Manipulation nicht zu verbergen.

Probieren Sie es selbst aus: Ändern Sie den Wert von `tool_name`, `agent_id` oder `timestamp` in der Zelle oben und führen Sie sie erneut aus. Jede Änderung erzeugt eine ungültige Quittung.


## Abschnitt 3: Kassenbons miteinander verketten

Ein einzelner Kassenbon schützt eine Aktion. Die meisten Agenten führen viele Aktionen aus. Um die gesamte Sequenz manipulationssicher zu machen, verknüpfen wir jeden Kassenbon mit dem vorherigen, indem wir den Hash des vorherigen Kassenbons in die Nutzlast des neuen Kassenbons aufnehmen.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Wenn jemand einen Kassenbon entfernt oder umordnet, bricht die Kette genau an dieser Stelle. Die Überprüfung eines späteren Kassenbons schlägt fehl, weil sein `previous_receipt_hash` nicht mehr mit dem tatsächlichen Hash seines Vorgängers übereinstimmt.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Durchbrechen Sie nun die Kette, indem Sie die mittlere Quittung manipulieren und erneut überprüfen. Die manipulierte Quittung schlägt bei der Signaturprüfung fehl, UND die nächste Quittung schlägt bei der Prüfung der Kettenverknüpfung fehl (weil ihr `previous_receipt_hash` nicht mehr mit dem geänderten Hash der mittleren Quittung übereinstimmt).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Quittung 0 wird immer noch verifiziert (sie wurde nicht verändert und hat keinen Vorgänger, von dem sie abhängig ist). Quittung 1 schlägt bei der Signaturprüfung fehl, weil wir `tool_args_hash` geändert haben. Quittung 2 schlägt bei der Kettenverknüpfungsprüfung fehl, weil ihr `previous_receipt_hash` gegen die ursprüngliche (jetzt modifizierte) Quittung 1 berechnet wurde.

Selbst wenn ein Angreifer Quittung 1 erneut signiert (was er ohne den privaten Schlüssel nicht tun kann), würde die Kettenverknüpfungsabweichung in Quittung 2 die Manipulation weiterhin aufdecken. Um die Änderung zu verbergen, müsste der Angreifer jede Quittung ab dem Änderungszeitpunkt neu signieren, wofür er den privaten Schlüssel besitzen muss.


## Abschnitt 4: Einen Agenten-Toolaufruf mit Quittungsunterzeichnung umhüllen

In einer echten Bereitstellung möchten Sie nicht, dass jeder Agentenautor daran denken muss, `make_receipt` aufzurufen. Die Quittungsunterzeichnung soll bei jedem Werkzeugaufruf automatisch erfolgen.

Hier ist das einfachste Muster: eine Wrapper-Klasse, die jede aufrufbare Tool-Funktion übernimmt und eine quittungserstellende Version davon zurückgibt. Dies passt sich jedem Agenten-Framework an, einschließlich des Microsoft Agent Framework (`agent_framework.foundry`).

Wenn Sie kein Microsoft Foundry-Projekt eingerichtet haben, zeigt das untenstehende lokale Mock dennoch das Muster.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integration mit dem Microsoft Agent Framework

Der obige `ReceiptedTool`-Wrapper ist framework-unabhängig. Um ihn in einem mit dem Microsoft Agent Framework erstellten Agenten zu verwenden, registrieren Sie die umschlossene Funktion als Werkzeug. Eine Skizze (Sie würden den Mock durch eine echte Microsoft Foundry Werkzeugregistrierung ersetzen):

```python
# Pseudocode, der die Integrationsform zeigt.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     anweisungen="Sie sind ein Contoso Travel-Agent ...",
#     werkzeuge=[receipted_lookup],   # das umschlossene Werkzeug, nicht die rohe Funktion
# )
# antwort = agent.run("Finde Flüge von Sydney nach Los Angeles im Juni.")
#
# # Nach dem Lauf hat jeder Werkzeugaufruf, den der Agent gemacht hat, eine signierte Quittung:
# audit_chain = receipted_lookup.receipts
```

Das Agent Framework muss nichts über Belege wissen. Die Belegsignierung wird um das Werkzeug herum eingebunden, nicht ins Framework integriert. So fügen Sie bestehendem Agenten-Code eine Herkunft hinzu, ohne den Agenten neu schreiben zu müssen.


## Rückblick und Stretch-Challenge

Du hast:

- Ein Ed25519-Schlüsselpaar generiert.
- Einen Beleg für einen Agentenwerkzeugaufruf erstellt und signiert.
- Den Beleg offline nur mit dem öffentlichen Schlüssel überprüft.
- Einen Beleg manipuliert und die fehlgeschlagene Überprüfung beobachtet.
- Eine hash-verkettete Sequenz aus drei Belegen aufgebaut.
- Die Mitte der Kette manipuliert und sowohl Signatur- als auch Kettenlink-Fehler beobachtet.
- Eine Werkzeugfunktion mit automatischer Belegsignierung umschlossen.

**Stretch-Challenge.** Erweitere das Belegschema um ein Feld `request_id` (eine UUID für verteiltes Tracing). Aktualisiere `make_receipt`, um dieses einzubeziehen, und bestätige, dass Belege weiterhin Ende-zu-Ende verifiziert werden. Ändere anschließend das Feld nach der Signierung und bestätige, dass die Überprüfung fehlschlägt. Dies zwingt dich dazu, zu verinnerlichen, wie jedes Byte der kanonischen Kodierung zur Signatur beiträgt.

**Wichtige Grenze.** Belege beweisen drei Dinge und nur drei Dinge: Zuordnung (dieser Schlüssel hat diesen Inhalt signiert), Integrität (der Inhalt hat sich seit der Signierung nicht geändert), und Reihenfolge (dieser Beleg kam nach jenem Beleg). Sie beweisen NICHT, dass die Aktion des Agenten korrekt war, dass die in `policy_id` benannte Richtlinie tatsächlich ausgewertet wurde oder dass der Agent jede Regel befolgt hat. Belege sind eine Grundlage. Governance ist das System, das du darauf aufbaust.

Lies die Lektion README erneut mit dieser Grenze im Hinterkopf. Der häufigste Fehler von Teams mit Belegen ist anzunehmen, dass "wir haben Belege" gleichbedeutend mit "wir sind reglementiert" ist. Das ist es nicht. Belege machen das Verhalten von Agenten prüfbar. Sie machen es nicht korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
